# Input Files Deep Dive (Toy Network + OD + FAF5)

This notebook inventories and inspects:
- Toy OD parquet files
- Toy FAF5 network GeoParquet files
- Input OD CSV files

For each file it reports:
- File type and path
- Existence and size
- Exact headers (ordered)
- Rows × columns
- Column dtypes and null counts
- Content preview (head/tail or full if small)
- Matrix-style OD summary where applicable
- Parquet schema details for `.pq` / `.gpq` files

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    import geopandas as gpd
except Exception:
    gpd = None

try:
    import pyarrow.parquet as pq
except Exception:
    pq = None

ROOT = Path.cwd()
print(f'Workspace root: {ROOT}')

TARGET_FILES = {
    'toy_od_parquet': [
        ROOT / 'sandbox/fairfax_soge_clusters_toy/inputs/census_datasets/faf5_od_matrix.pq',
        ROOT / 'sandbox/fairfax_baseline/inputs/test_17node/faf5_od_matrix_17x17_test.pq',
    ],
    'toy_faf5_network_gpq': [
        ROOT / 'sandbox/fairfax_soge_clusters_toy/inputs/networks/faf5/faf5_road_links.gpq',
        ROOT / 'sandbox/fairfax_baseline/inputs/networks/faf5/faf5_road_links.gpq',
    ],
    'input_od_csv': [
        ROOT / 'sandbox/fairfax_baseline/inputs/test_17node/faf5_od_matrix_17x17_test.csv',
        ROOT / 'sandbox/fairfax_soge_clusters_toy/inputs/census_datasets/faf5_od_node_mapping.csv',
    ],
}

for group, paths in TARGET_FILES.items():
    print(f'\n[{group}]')
    for p in paths:
        status = 'FOUND' if p.exists() else 'MISSING'
        print(f' - {status}: {p}')

In [ ]:
def human_size(n):
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    i = 0
    x = float(n)
    while x >= 1024 and i < len(units) - 1:
        x /= 1024
        i += 1
    return f'{x:.2f} {units[i]}'

def parquet_schema_string(path):
    if pq is None:
        return 'pyarrow not available'
    try:
        return str(pq.read_schema(path))
    except Exception as e:
        return f'Could not read parquet schema: {e}'

def read_table(path):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.pq', '.parquet'}:
        return pd.read_parquet(path)
    if suffix == '.gpq':
        if gpd is not None:
            return gpd.read_parquet(path)
        return pd.read_parquet(path)
    raise ValueError(f'Unsupported file type: {path}')

def summarize_od_matrix(df):
    cols = [str(c) for c in df.columns]
    lc = {c.lower(): c for c in cols}

    origin_col = lc.get('origin_node') or lc.get('origin') or lc.get('o')
    dest_col = lc.get('destination_node') or lc.get('destination') or lc.get('d')

    flow_candidates = [
        c for c in df.columns
        if (('flow' in str(c).lower()) or str(c).lower().startswith('car'))
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    if origin_col and dest_col:
        n_o = df[origin_col].nunique(dropna=True)
        n_d = df[dest_col].nunique(dropna=True)
        possible = n_o * n_d if n_o and n_d else np.nan
        density = (len(df) / possible) if possible and possible > 0 else np.nan

        print('OD Matrix Summary:')
        print(f' - Origin column: {origin_col}')
        print(f' - Destination column: {dest_col}')
        print(f' - Unique origins: {n_o}')
        print(f' - Unique destinations: {n_d}')
        print(f' - Possible O-D pairs: {possible}')
        print(f' - Observed O-D rows: {len(df)}')
        if pd.notna(density):
            print(f' - Matrix density: {density:.4f}')

        if flow_candidates:
            fcol = flow_candidates[0]
            print(f' - Flow-like column used for summary: {fcol}')
            print(f' - Total {fcol}: {df[fcol].sum()}')
            print(f' - Non-zero {fcol} rows: {(df[fcol] != 0).sum()}')

            pivot = (
                df.pivot_table(index=origin_col, columns=dest_col, values=fcol, aggfunc='sum', fill_value=0)
            )
            print(f' - Pivot matrix shape ({fcol}): {pivot.shape[0]} x {pivot.shape[1]}')
            if pivot.shape[0] <= 25 and pivot.shape[1] <= 25:
                print(' - Pivot matrix values:')
                print(pivot)
    else:
        # Heuristic: check if this is a square-ish numeric matrix CSV
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(df) > 0 and len(numeric_cols) > 0:
            print('Matrix-like Summary (heuristic):')
            print(f' - Numeric columns: {len(numeric_cols)} / {len(df.columns)}')
            print(f' - Dataframe shape: {df.shape[0]} x {df.shape[1]}')

def inspect_file(path):
    print('\n' + '=' * 120)
    print(f'FILE: {path}')
    print('=' * 120)

    if not path.exists():
        print('Status: MISSING')
        return

    suffix = path.suffix.lower()
    print(f'File type (extension): {suffix}')
    print(f'Size: {path.stat().st_size} bytes ({human_size(path.stat().st_size)})')

    if suffix in {'.pq', '.parquet', '.gpq'}:
        print('Parquet schema:')
        print(parquet_schema_string(path))

    df = read_table(path)

    print('\nTable shape:')
    print(f' - Rows x Columns: {df.shape[0]} x {df.shape[1]}')

    print('\nExact headers (ordered):')
    for i, c in enumerate(df.columns, start=1):
        print(f' {i:>3}. {c}')

    print('\nDtypes:')
    print(df.dtypes)

    print('\nNull counts:')
    print(df.isna().sum())

    print('\nContent preview:')
    if df.shape[0] <= 60 and df.shape[1] <= 20:
        print(df)
    else:
        print('HEAD (first 20 rows):')
        print(df.head(20))
        print('\nTAIL (last 20 rows):')
        print(df.tail(20))

    print('\nDescriptive stats (numeric columns):')
    num = df.select_dtypes(include='number')
    if num.shape[1] == 0:
        print(' - No numeric columns')
    else:
        print(num.describe(include='all').T)

    summarize_od_matrix(df)

In [ ]:
for group, paths in TARGET_FILES.items():
    print(f'\n\n### GROUP: {group} ###')
    for p in paths:
        inspect_file(p)

In [ ]:
# --- Script 1 output check: total OD flow + crude flow-distance decay test ---
from pathlib import Path
import ast
import numpy as np
import pandas as pd

# Resolve repository root robustly (works whether cwd is repo root or sandbox/)
cwd = Path.cwd()
ROOT = cwd
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

base_results = ROOT / "sandbox/results/base_scenario/revision"
odpfc_path = base_results / "odpfc.pq"
edge_path = base_results / "edge_flows.gpq"

print(f"Workspace/root detected: {ROOT}")
print(f"ODPFC path: {odpfc_path}")
print(f"Edge flows path: {edge_path}")

if not odpfc_path.exists() or not edge_path.exists():
    raise FileNotFoundError("Expected script-1 outputs not found under sandbox/results/base_scenario/revision")

# Read outputs
odpfc = pd.read_parquet(odpfc_path)
try:
    import geopandas as gpd
    edges = gpd.read_parquet(edge_path)
except Exception:
    edges = pd.read_parquet(edge_path)

# Robust parser for path-like values stored as list/tuple/array/string

def to_edge_id_list(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(v) for v in value]
    if isinstance(value, np.ndarray):
        vals = value.tolist()
        if vals and all(isinstance(v, str) and len(v) == 1 for v in vals):
            s = "".join(vals)
            if s.startswith("[") and s.endswith("]"):
                try:
                    return [str(v) for v in ast.literal_eval(s)]
                except Exception:
                    pass
        return [str(v) for v in vals]
    if isinstance(value, str):
        s = value.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set, np.ndarray)):
                    return [str(v) for v in parsed]
            except Exception:
                pass
        return [tok.strip() for tok in s.split(",") if tok.strip()]
    return [str(value)]

# Pick flow column
flow_col = "flow" if "flow" in odpfc.columns else ("Car21" if "Car21" in odpfc.columns else None)
if flow_col is None:
    raise KeyError("Could not find flow column in odpfc (expected 'flow' or 'Car21').")

# Total OD flow
total_od_flow = float(odpfc[flow_col].sum())
print(f"\nTotal OD flow ({flow_col}): {total_od_flow:,.2f}")
print(f"OD rows: {len(odpfc):,}")

# Build edge length (miles) dictionary
edges = edges.copy()
edges["e_id"] = edges["e_id"].astype(str)
if "length_mile" in edges.columns:
    edge_len_miles = edges.set_index("e_id")["length_mile"].astype(float).to_dict()
elif "length" in edges.columns:
    # assume meters if only 'length' exists
    edge_len_miles = (edges.set_index("e_id")["length"].astype(float) / 1609.34).to_dict()
else:
    raise KeyError("Could not find 'length_mile' or 'length' in edge_flows output.")

# Normalize IDs + parse paths
odpfc = odpfc.copy()
odpfc["path_list"] = odpfc["path"].apply(to_edge_id_list)

# Compute route distance (sum of edge miles on each path)
def path_distance_miles(path_list):
    return float(sum(edge_len_miles.get(str(e), 0.0) for e in path_list))

odpfc["distance_miles"] = odpfc["path_list"].apply(path_distance_miles)

# Basic sanity
valid = odpfc[(odpfc[flow_col] > 0) & (odpfc["distance_miles"] > 0)].copy()
print(f"Valid rows for decay test (flow>0 & distance>0): {len(valid):,}")

if len(valid) < 5:
    print("Not enough valid rows to evaluate flow-distance relationship.")
else:
    # Crude decay metrics
    corr_linear = float(valid[["distance_miles", flow_col]].corr().iloc[0, 1])
    valid["log_flow"] = np.log1p(valid[flow_col].astype(float))
    corr_log = float(valid[["distance_miles", "log_flow"]].corr().iloc[0, 1])

    # slope of log(flow) ~ distance (negative suggests decay)
    slope, intercept = np.polyfit(valid["distance_miles"].astype(float), valid["log_flow"].astype(float), 1)

    print("\nCrude flow-distance decay test:")
    print(f" - Corr(distance, flow): {corr_linear:.4f}")
    print(f" - Corr(distance, log1p(flow)): {corr_log:.4f}")
    print(f" - Slope log1p(flow) ~ distance: {slope:.6f} (negative => decay)")

    # Distance-bin summary
    valid["dist_bin"] = pd.qcut(valid["distance_miles"], q=5, duplicates="drop")
    b = valid.groupby("dist_bin", observed=True).agg(
        n=(flow_col, "size"),
        mean_distance_mi=("distance_miles", "mean"),
        mean_flow=(flow_col, "mean"),
        median_flow=(flow_col, "median"),
        total_flow=(flow_col, "sum"),
    )
    print("\nFlow by distance quintile (quick check):")
    print(b.to_string())

Workspace/root detected: c:\Users\alimu\Desktop\Github\DAFNI-NIRD-clean
ODPFC path: c:\Users\alimu\Desktop\Github\DAFNI-NIRD-clean\sandbox\results\base_scenario\revision\odpfc.pq
Edge flows path: c:\Users\alimu\Desktop\Github\DAFNI-NIRD-clean\sandbox\results\base_scenario\revision\edge_flows.gpq

Total OD flow (flow): 10,032.00
OD rows: 272
Valid rows for decay test (flow>0 & distance>0): 272

Crude flow-distance decay test:
 - Corr(distance, flow): -0.8421
 - Corr(distance, log1p(flow)): -0.9398
 - Slope log1p(flow) ~ distance: -0.043553 (negative => decay)

Flow by distance quintile (quick check):
                   n  mean_distance_mi  mean_flow  median_flow  total_flow
dist_bin                                                                  
(0.862, 8.885]    55          6.021649  68.581818         65.0      3772.0
(8.885, 15.309]   54         12.487634  41.481481         38.0      2240.0
(15.309, 21.095]  55         18.143611  30.290909         29.0      1666.0
(21.095, 28.765]  

: 